In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Customer Churn Project Started!")


In [ ]:
pip install seaborn

In [ ]:

import os

print(os.getcwd())
print(os.listdir())


In [ ]:
import pandas as pd

df = pd.read_csv("Telco-Customer-Churn.csv")

df.head()

In [ ]:
print("Dataset shape:", df.shape)
print(df.columns.tolist())

In [ ]:

# Check dataset information
df.info()

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())

# Check summary statistics
df.describe()


In [ ]:

# Check blank values in TotalCharges
print("Blank TotalCharges:",
      (df["TotalCharges"].str.strip() == "").sum())

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

# Check missing values after conversion
print("Missing TotalCharges:",
      df["TotalCharges"].isnull().sum())

# Check data types
print(df[["tenure", "MonthlyCharges", "TotalCharges"]].dtypes)


In [ ]:

# Inspect customers with missing TotalCharges
df.loc[
    df["TotalCharges"].isnull(),
    ["customerID", "tenure", "MonthlyCharges",
     "TotalCharges", "Churn"]
]


In [ ]:

# Fill TotalCharges with 0 for new customers
df.loc[
    (df["tenure"] == 0) & (df["TotalCharges"].isnull()),
    "TotalCharges"
] = 0

# Verify missing values
print("Missing TotalCharges:", df["TotalCharges"].isnull().sum())

# Check dataset shape
print("Dataset shape:", df.shape)

# Check data types
print(df[["tenure", "MonthlyCharges", "TotalCharges"]].dtypes)


In [ ]:

# Count customers who churned and stayed
print(df["Churn"].value_counts())

# Calculate churn percentage
churn_percent = df["Churn"].value_counts(normalize=True) * 100

print("\nChurn Percentage:")
print(churn_percent.round(2))

# Visualize churn distribution
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn Status")
plt.ylabel("Number of Customers")
plt.show()


In [ ]:

# Churn rate by contract type
contract_churn = pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
) * 100

print(contract_churn.round(2))

# Visualize churn by contract
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Customer Churn by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")
plt.xticks(rotation=15)
plt.show()


In [ ]:

# Compare monthly charges by churn status
print(df.groupby("Churn")["MonthlyCharges"].agg(
    ["mean", "median"]
).round(2))

# Visualize the distribution
sns.barplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges vs Customer Churn")
plt.xlabel("Churn Status")
plt.ylabel("Monthly Charges")
plt.show()


In [ ]:

# Churn rate by internet service
internet_churn = pd.crosstab(
    df["InternetService"],
    df["Churn"],
    normalize="index"
) * 100

print(internet_churn.round(2))

# Visualize churn by internet service
sns.countplot(
    data=df,
    x="InternetService",
    hue="Churn"
)

plt.title("Customer Churn by Internet Service")
plt.xlabel("Internet Service")
plt.ylabel("Number of Customers")
plt.show()


In [ ]:

# Churn rate by payment method
payment_churn = pd.crosstab(
    df["PaymentMethod"],
    df["Churn"],
    normalize="index"
) * 100

print(payment_churn.round(2))

# Visualize churn rates
payment_churn["Yes"].sort_values().plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Churn Rate by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


In [ ]:

# Create a copy for machine learning
ml_df = df.copy()

# Remove customer ID
ml_df = ml_df.drop(columns=["customerID"])

# Check target distribution
print("Dataset shape:", ml_df.shape)
print("\nTarget distribution:")
print(ml_df["Churn"].value_counts())

# Check data types and missing values
print("\nMissing values:", ml_df.isnull().sum().sum())
print("\nData types:")
print(ml_df.dtypes)


In [ ]:
pip install scikit-learn

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Separate features and target
X = ml_df.drop(columns=["Churn"])
y = ml_df["Churn"].map({"No": 0, "Yes": 1})

# Identify categorical and numerical columns
cat_cols = X.select_dtypes(include=["str", "object"]).columns
num_cols = X.select_dtypes(include=["number"]).columns

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Preprocess numerical and categorical features
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

# Build pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

print("Model trained successfully!")
print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Stayed", "Churned"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:

# Train an improved Logistic Regression model
balanced_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

balanced_model.fit(X_train, y_train)

# Predictions
balanced_pred = balanced_model.predict(X_test)

print("Improved Model Results:\n")
print(classification_report(
    y_test,
    balanced_pred,
    target_names=["Stayed", "Churned"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, balanced_pred))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Results:\n")
print(classification_report(
    y_test,
    rf_pred,
    target_names=["Stayed", "Churned"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

In [ ]:
# Get feature names after preprocessing
feature_names = (
    balanced_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

# Get Logistic Regression coefficients
coefficients = (
    balanced_model.named_steps["classifier"].coef_[0]
)

# Create feature importance table
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

feature_importance["Importance"] = (
    feature_importance["Coefficient"].abs()
)

feature_importance = feature_importance.sort_values(
    by="Importance", ascending=False
)

print(feature_importance.head(15))

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Select customer behaviour and billing features
segment_df = df[
    ["tenure", "MonthlyCharges", "TotalCharges"]
].copy()

# Scale features so different ranges don't dominate
scaler = StandardScaler()
segment_scaled = scaler.fit_transform(segment_df)

print("Segmentation data shape:", segment_scaled.shape)
print("Missing values:", segment_df.isnull().sum().sum())
print("Segmentation data prepared successfully!")

In [ ]:
inertia_values = []

for k in range(1, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    kmeans.fit(segment_scaled)
    inertia_values.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertia_values, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method - Customer Segmentation")
plt.xticks(range(1, 11))
plt.grid(True)
plt.show()

In [ ]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

df["CustomerSegment"] = kmeans.fit_predict(segment_scaled)

print("Customer Segment Counts:")
print(df["CustomerSegment"].value_counts().sort_index())

In [ ]:
segment_summary = df.groupby("CustomerSegment").agg(
    Customer_Count=("customerID", "count"),
    Avg_Tenure=("tenure", "mean"),
    Avg_Monthly_Charges=("MonthlyCharges", "mean"),
    Avg_Total_Charges=("TotalCharges", "mean"),
    Churn_Rate=("Churn", lambda x: (x == "Yes").mean() * 100)
).round(2)

print(segment_summary)

In [ ]:
segment_summary["Churn_Rate"].plot(
    kind="bar",
    figsize=(8, 5),
    color="tomato"
)

plt.title("Churn Rate by Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Select one customer from the test data
sample_customer = X_test.iloc[[0]]

# Predict churn using our balanced Logistic Regression model
prediction = balanced_model.predict(sample_customer)[0]
probability = balanced_model.predict_proba(sample_customer)[0][1]

print("Predicted Churn:", "Yes" if prediction == 1 else "No")
print(f"Churn Probability: {probability:.2%}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Get churn probabilities
churn_probabilities = balanced_model.predict_proba(X_test)[:, 1]

# Use a 0.40 threshold
threshold = 0.40
threshold_pred = (churn_probabilities >= threshold).astype(int)

print("Threshold:", threshold)
print("\nClassification Report:")
print(classification_report(
    y_test,
    threshold_pred,
    target_names=["Stayed", "Churned"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, threshold_pred))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Churn distribution
df["Churn"].value_counts().plot(
    kind="bar", ax=axes[0]
)
axes[0].set_title("Customer Churn Distribution")
axes[0].set_xlabel("Churn")
axes[0].set_ylabel("Number of Customers")
axes[0].tick_params(axis="x", rotation=0)

# 2. Churn by contract
contract_churn = pd.crosstab(
    df["Contract"], df["Churn"], normalize="index"
) * 100

contract_churn["Yes"].plot(
    kind="bar", ax=axes[1]
)
axes[1].set_title("Churn Rate by Contract")
axes[1].set_xlabel("Contract Type")
axes[1].set_ylabel("Churn Rate (%)")
axes[1].tick_params(axis="x", rotation=20)

# 3. Churn by customer segment
segment_summary["Churn_Rate"].plot(
    kind="bar", ax=axes[2]
)
axes[2].set_title("Churn Rate by Customer Segment")
axes[2].set_xlabel("Customer Segment")
axes[2].set_ylabel("Churn Rate (%)")
axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()